# LogFile Exploration - LoneWolf Dataset

## Objective
Analyze LogFile structure, parse timestamp changes from Detail column, and understand Oh et al.'s LogFile-based detection methodology.

## Dataset
LoneWolf LogFile parsed by Oh et al.'s NTFS Log Tracker tool



In [1]:
## Cell 1: Setup and Load Data

import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import datetime

# Paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
LOGFILE_PATH = BASE_DIR / 'data/validation/logfile/LoneWolf-LogFile.csv'
SUSPICIOUS_PATH = BASE_DIR / 'data/validation/suspicious/LoneWolf-Suspicious.csv'

# Load LogFile
logfile_df = pd.read_csv(LOGFILE_PATH)

print(f"LogFile Records: {len(logfile_df):,}")
print(f"\nColumns: {list(logfile_df.columns)}")
print(f"\n$MFT timestamps already included:")
print(f"  - CreationTime: {logfile_df['CreationTime'].notna().sum():,} non-null")
print(f"  - ModifiedTime: {logfile_df['ModifiedTime'].notna().sum():,} non-null")
print(f"  - MFTModifiedTime: {logfile_df['MFTModifiedTime'].notna().sum():,} non-null")
print(f"  - AccessedTime: {logfile_df['AccessedTime'].notna().sum():,} non-null")


LogFile Records: 16,882

Columns: ['LSN', 'EventTime(UTC+8)', 'Event', 'Detail', 'File/Directory Name', 'Full Path', 'CreationTime', 'ModifiedTime', 'MFTModifiedTime', 'AccessedTime', 'Redo', 'Target VCN', 'Cluster Index']

$MFT timestamps already included:
  - CreationTime: 6,590 non-null
  - ModifiedTime: 6,590 non-null
  - MFTModifiedTime: 6,590 non-null
  - AccessedTime: 6,590 non-null


In [2]:
# Cell 2: Analyze Event Types
# Count event types
print("Event Type Distribution:")
print(logfile_df['Event'].value_counts().head(20))

# Check for timestamp-related events
timestamp_events = logfile_df[logfile_df['Event'].str.contains('Updating', na=False)]
print(f"\n\nTimestamp Update Events: {len(timestamp_events):,}")
print(timestamp_events['Event'].value_counts())

# Check for creation events
creation_events = logfile_df[logfile_df['Event'].str.contains('Creation', na=False)]
print(f"\n\nFile Creation Events: {len(creation_events):,}")
print(creation_events['Event'].value_counts())


Event Type Distribution:
Event
Updating Modified Time                                4474
File Deletion                                         2259
File Creation                                         2126
Writing Content of Non-Resident File                  1803
Updating MFTModified Time                             1287
Renaming File                                          813
Writing Content of Resident File                       700
# Check Point                                          578
Time Reversal Event                                    573
File Creation(File System Tunneling)                   439
Move(After)                                            282
Move(Before)                                           271
Changing FileAttribute                                  74
Directory Deletion                                      49
Directory Creation                                      38
Updating MFTModified Time & Changing FileAttribute      15
Time Reversal Event & Cha

In [3]:
# Cell 3: Parse Detail COlumn for Tiemstamp Changes 
# Function to parse timestamp changes from Detail column
def parse_timestamp_change(detail_str):
    """
    Parse Detail column to extract before and after timestamps
    Example: "ModifiedTime : 2018-04-06 15:05:36 -> 2018-04-06 15:05:37"
    Returns: (timestamp_type, before, after)
    """
    if pd.isna(detail_str):
        return None, None, None
    
    # Pattern: "TimestampType : YYYY-MM-DD HH:MM:SS -> YYYY-MM-DD HH:MM:SS"
    pattern = r'(\w+Time)\s*:\s*(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s*->\s*(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})'
    match = re.search(pattern, detail_str)
    
    if match:
        timestamp_type = match.group(1)
        before = match.group(2)
        after = match.group(3)
        return timestamp_type, before, after
    
    return None, None, None

# Apply to timestamp update events
timestamp_events = logfile_df[logfile_df['Event'].str.contains('Updating', na=False)].copy()
timestamp_events[['TS_Type', 'TS_Before', 'TS_After']] = timestamp_events['Detail'].apply(
    lambda x: pd.Series(parse_timestamp_change(x))
)

print(f"Parsed {timestamp_events['TS_Type'].notna().sum():,} timestamp changes")
print(f"\nTimestamp types changed:")
print(timestamp_events['TS_Type'].value_counts())

# Show examples
print(f"\nSample timestamp changes:")
print(timestamp_events[timestamp_events['TS_Type'].notna()][
    ['EventTime(UTC+8)', 'Event', 'File/Directory Name', 'TS_Type', 'TS_Before', 'TS_After']
].head(10).to_string())


Parsed 5,789 timestamp changes

Timestamp types changed:
TS_Type
ModifiedTime       4487
MFTModifiedTime    1302
Name: count, dtype: int64

Sample timestamp changes:
         EventTime(UTC+8)                   Event                 File/Directory Name       TS_Type            TS_Before             TS_After
1                     NaN  Updating Modified Time        20988bb0bb16e7d1_0 <Guessed>  ModifiedTime  2018-04-06 15:05:36  2018-04-06 15:05:37
2                     NaN  Updating Modified Time        20988bb0bb16e7d1_0 <Guessed>  ModifiedTime  2018-04-06 15:05:37  2018-04-06 15:05:38
5   04/06/18 15:05:39:539  Updating Modified Time               cloud_graph <Guessed>  ModifiedTime  2018-04-06 15:05:33  2018-04-06 15:05:39
6   04/06/18 15:05:39:539  Updating Modified Time            cloud_graph.db <Guessed>  ModifiedTime  2018-04-06 15:05:33  2018-04-06 15:05:39
8                     NaN  Updating Modified Time        20988bb0bb16e7d1_0 <Guessed>  ModifiedTime  2018-04-06 15:05:38  20

In [4]:
# Cell 4: Detect Suspicious Timestamp Changes
# Oh et al.'s LogFile detection criteria:
# 1. CreationTime changed (SI-C modification)
# 2. ModifiedTime decreased (moved backward in time)
# 3. Check for zero nanoseconds

# Filter for CreationTime changes
creation_time_changes = timestamp_events[timestamp_events['TS_Type'] == 'CreationTime'].copy()
print(f"CreationTime Change Events: {len(creation_time_changes):,}")

# Parse timestamps
creation_time_changes['TS_Before_Parsed'] = pd.to_datetime(
    creation_time_changes['TS_Before'], 
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)
creation_time_changes['TS_After_Parsed'] = pd.to_datetime(
    creation_time_changes['TS_After'],
    format='%Y-%m-%d %H:%M:%S', 
    errors='coerce'
)

# Calculate time difference
creation_time_changes['Time_Diff_Seconds'] = (
    creation_time_changes['TS_After_Parsed'] - creation_time_changes['TS_Before_Parsed']
).dt.total_seconds()

# Flag suspicious: CreationTime changed to the past (negative diff)
creation_time_changes['Changed_To_Past'] = creation_time_changes['Time_Diff_Seconds'] < 0

# Check for zero nanoseconds (timestamps ending in :00)
creation_time_changes['Zero_Seconds_After'] = creation_time_changes['TS_After'].str.endswith(':00')

# Mark suspicious
creation_time_changes['Suspicious'] = (
    creation_time_changes['Changed_To_Past'] | 
    creation_time_changes['Zero_Seconds_After']
)

print(f"\nSuspicious CreationTime changes:")
print(f"  Changed to past: {creation_time_changes['Changed_To_Past'].sum():,}")
print(f"  Zero seconds (rounded): {creation_time_changes['Zero_Seconds_After'].sum():,}")
print(f"  Total suspicious: {creation_time_changes['Suspicious'].sum():,}")

# Show suspicious examples
if creation_time_changes['Suspicious'].sum() > 0:
    print(f"\nSuspicious CreationTime Changes:")
    suspicious = creation_time_changes[creation_time_changes['Suspicious']][
        ['EventTime(UTC+8)', 'File/Directory Name', 'TS_Before', 'TS_After', 'Time_Diff_Seconds']
    ]
    print(suspicious.head(10).to_string())


CreationTime Change Events: 0

Suspicious CreationTime changes:
  Changed to past: 0
  Zero seconds (rounded): 0
  Total suspicious: 0


In [5]:
# Cell 5: Check ModifiedTime Changes 
# Filter for ModifiedTime changes
modified_time_changes = timestamp_events[timestamp_events['TS_Type'] == 'ModifiedTime'].copy()
print(f"ModifiedTime Change Events: {len(modified_time_changes):,}")

# Parse timestamps
modified_time_changes['TS_Before_Parsed'] = pd.to_datetime(
    modified_time_changes['TS_Before'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)
modified_time_changes['TS_After_Parsed'] = pd.to_datetime(
    modified_time_changes['TS_After'],
    format='%Y-%m-%d %H:%M:%S',
    errors='coerce'
)

# Calculate time difference
modified_time_changes['Time_Diff_Seconds'] = (
    modified_time_changes['TS_After_Parsed'] - modified_time_changes['TS_Before_Parsed']
).dt.total_seconds()

# Flag suspicious: ModifiedTime decreased (Oh et al.'s criterion)
modified_time_changes['Decreased'] = modified_time_changes['Time_Diff_Seconds'] < 0

print(f"\nModifiedTime decreased (moved to past): {modified_time_changes['Decreased'].sum():,}")

if modified_time_changes['Decreased'].sum() > 0:
    print(f"\nSuspicious ModifiedTime Changes (decreased):")
    suspicious = modified_time_changes[modified_time_changes['Decreased']][
        ['EventTime(UTC+8)', 'File/Directory Name', 'TS_Before', 'TS_After', 'Time_Diff_Seconds']
    ]
    print(suspicious.head(10).to_string())


ModifiedTime Change Events: 4,487

ModifiedTime decreased (moved to past): 0


In [6]:
# Cell 6: Idenity Files with File Creation Events 
# Find File Creation events
creation_events = logfile_df[
    logfile_df['Event'].str.contains('File Creation', na=False)
].copy()

print(f"File Creation Events: {len(creation_events):,}")

# Parse creation timestamps
creation_events['EventTime_Parsed'] = pd.to_datetime(
    creation_events['EventTime(UTC+8)'],
    format='%m/%d/%y %H:%M:%S:%f',
    errors='coerce'
)

creation_events['SI_CreationTime_Parsed'] = pd.to_datetime(
    creation_events['CreationTime'],
    format='%m/%d/%y %H:%M:%S:%f',
    errors='coerce'
)

# Calculate difference between event time and SI-C
creation_events['Creation_Diff_Seconds'] = abs(
    (creation_events['SI_CreationTime_Parsed'] - creation_events['EventTime_Parsed']).dt.total_seconds()
)

# Oh et al.: If File Creation exists, check if SI-C differs from creation event time
creation_events['Anomalous_Creation'] = creation_events['Creation_Diff_Seconds'] > 5

print(f"\nFile Creation Anomalies (diff > 5s): {creation_events['Anomalous_Creation'].sum():,}")

# Show examples
print(f"\nSample File Creation Events:")
print(creation_events[
    ['EventTime(UTC+8)', 'File/Directory Name', 'Event', 'CreationTime', 'Creation_Diff_Seconds']
].head(10).to_string())

if creation_events['Anomalous_Creation'].sum() > 0:
    print(f"\nAnomalous File Creations:")
    print(creation_events[creation_events['Anomalous_Creation']][
        ['EventTime(UTC+8)', 'File/Directory Name', 'CreationTime', 'Creation_Diff_Seconds']
    ].head(10).to_string())


File Creation Events: 2,565

File Creation Anomalies (diff > 5s): 424

Sample File Creation Events:
         EventTime(UTC+8)                       File/Directory Name                                 Event           CreationTime  Creation_Diff_Seconds
3   04/06/18 15:05:39:539                    cloud_graph.db-journal  File Creation(File System Tunneling)  04/06/18 15:05:33:533                  6.006
13  04/06/18 15:05:41:541  9a30f7b8-389c-4cf2-9b40-d0123e980fd5.tmp                         File Creation  04/06/18 15:05:41:541                  0.000
16  04/06/18 15:05:41:541  c8b05115-0844-433f-9d27-d5bad54a48c7.tmp                         File Creation  04/06/18 15:05:41:541                  0.000
20  04/06/18 15:05:41:541                Local State~RF305a3384.TMP                         File Creation  04/06/18 15:05:41:541                  0.000
27  04/06/18 15:05:41:541                Preferences~RF305a3394.TMP                         File Creation  04/06/18 15:05:41:541            

In [7]:
# Cell 7: Check for File System Tunneling 
# Oh et al. already flags tunneling in some events
tunneling_events = logfile_df[
    logfile_df['Event'].str.contains('Tunneling', na=False)
].copy()

print(f"File System Tunneling Events (pre-flagged): {len(tunneling_events):,}")

if len(tunneling_events) > 0:
    print(f"\nTunneling Events:")
    print(tunneling_events[
        ['EventTime(UTC+8)', 'Event', 'File/Directory Name', 'Full Path']
    ].head(10).to_string())

# Identify deletion events (for tunneling pattern detection)
deletion_events = logfile_df[
    logfile_df['Event'].str.contains('Deletion', na=False)
].copy()

print(f"\n\nFile Deletion Events: {len(deletion_events):,}")


File System Tunneling Events (pre-flagged): 439

Tunneling Events:
          EventTime(UTC+8)                                 Event     File/Directory Name                                                                                  Full Path
3    04/06/18 15:05:39:539  File Creation(File System Tunneling)  cloud_graph.db-journal  \Users\jcloudy\AppData\Local\Google\Drive\user_default\cloud_graph\cloud_graph.db-journal
78    04/06/18 15:06:02:62  File Creation(File System Tunneling)      sync_config.db-wal                  \Users\jcloudy\AppData\Local\Google\Drive\user_default\sync_config.db-wal
80    04/06/18 15:06:02:62  File Creation(File System Tunneling)      sync_config.db-shm                  \Users\jcloudy\AppData\Local\Google\Drive\user_default\sync_config.db-shm
110   04/06/18 15:06:04:64  File Creation(File System Tunneling)      sync_config.db-wal                  \Users\jcloudy\AppData\Local\Google\Drive\user_default\sync_config.db-wal
112   04/06/18 15:06:04:64  File 

In [8]:
# Load ground truth
ground_truth = pd.read_csv(SUSPICIOUS_PATH)

# Extract filenames from ground truth
gt_files = set()
for detail in ground_truth['detail']:
    if 'timestamp of ' in str(detail):
        filename = detail.split('timestamp of ')[1].split(' may')[0].strip('"')
        gt_files.add(filename)

print(f"Ground Truth Suspicious Files: {len(gt_files)}")
print(f"Files: {sorted(gt_files)}")

# Check if we found them in LogFile
logfile_files = set(logfile_df['File/Directory Name'].dropna().unique())

found_in_logfile = gt_files.intersection(logfile_files)
missing_from_logfile = gt_files - logfile_files

print(f"\n\nGround truth files found in LogFile: {len(found_in_logfile)}")
print(f"Files: {sorted(found_in_logfile)}")

print(f"\n\nGround truth files NOT in LogFile: {len(missing_from_logfile)}")
print(f"Files: {sorted(missing_from_logfile)}")

# For found files, check if we detected their timestamp changes
if len(found_in_logfile) > 0:
    print(f"\n\nChecking detection for ground truth files:")
    for filename in sorted(found_in_logfile):
        # Check CreationTime changes
        ct_changes = creation_time_changes[
            creation_time_changes['File/Directory Name'] == filename
        ]
        
        # Check ModifiedTime changes
        mt_changes = modified_time_changes[
            modified_time_changes['File/Directory Name'] == filename
        ]
        
        print(f"\n{filename}:")
        print(f"  CreationTime changes: {len(ct_changes)}")
        if len(ct_changes) > 0:
            print(f"    Suspicious: {ct_changes['Suspicious'].sum()}")
        print(f"  ModifiedTime changes: {len(mt_changes)}")
        if len(mt_changes) > 0:
            print(f"    Decreased: {mt_changes['Decreased'].sum()}")


Ground Truth Suspicious Files: 12
Files: ['AIRPORT INFORMATION.docx', 'BladeofGrass.jpg', 'CubaDearmed.jpg', 'DarkWolf.png', 'DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 'Huckleberry.png', 'MyTiredHead.jpg', 'Planning.docx', 'RedGuns.jpg', 'Sheep.jpg']


Ground truth files found in LogFile: 12
Files: ['AIRPORT INFORMATION.docx', 'BladeofGrass.jpg', 'CubaDearmed.jpg', 'DarkWolf.png', 'DeathToll.jpg', 'DemLogic.jpg', 'HoldMyTidePod.jpg', 'Huckleberry.png', 'MyTiredHead.jpg', 'Planning.docx', 'RedGuns.jpg', 'Sheep.jpg']


Ground truth files NOT in LogFile: 0
Files: []


Checking detection for ground truth files:

AIRPORT INFORMATION.docx:
  CreationTime changes: 0
  ModifiedTime changes: 0

BladeofGrass.jpg:
  CreationTime changes: 0
  ModifiedTime changes: 0

CubaDearmed.jpg:
  CreationTime changes: 0
  ModifiedTime changes: 0

DarkWolf.png:
  CreationTime changes: 0
  ModifiedTime changes: 0

DeathToll.jpg:
  CreationTime changes: 0
  ModifiedTime changes: 0

DemLogic.jpg:
  Cr

In [9]:
# Compare what we can detect in LogFile vs UsnJrnl
print("LogFile Detection Capabilities:")
print("================================")
print(f"Total records: {len(logfile_df):,}")
print(f"\nTimestamp change events:")
print(f"  CreationTime changes: {len(creation_time_changes):,}")
print(f"  ModifiedTime changes: {len(modified_time_changes):,}")
print(f"  Total timestamp changes: {len(timestamp_events):,}")

print(f"\nFile lifecycle events:")
print(f"  File Creation: {len(creation_events):,}")
print(f"  File Deletion: {len(deletion_events):,}")
print(f"  Tunneling (flagged): {len(tunneling_events):,}")

print(f"\nDetection summary:")
print(f"  Suspicious CreationTime changes: {creation_time_changes['Suspicious'].sum() if len(creation_time_changes) > 0 else 0:,}")
print(f"  Suspicious ModifiedTime changes: {modified_time_changes['Decreased'].sum() if len(modified_time_changes) > 0 else 0:,}")
print(f"  Anomalous File Creations: {creation_events['Anomalous_Creation'].sum() if len(creation_events) > 0 else 0:,}")

print("\n\nKey Findings:")
print("1. LogFile already has MACE timestamps from MFT (no enrichment needed)")
print("2. Timestamp changes are explicitly labeled in Event column")
print("3. Before/after values are in Detail column (need parsing)")
print("4. File system tunneling is partially pre-flagged")
print("5. LogFile complements UsnJrnl (different event coverage)")


LogFile Detection Capabilities:
Total records: 16,882

Timestamp change events:
  CreationTime changes: 0
  ModifiedTime changes: 4,487
  Total timestamp changes: 5,789

File lifecycle events:
  File Creation: 2,565
  File Deletion: 2,308
  Tunneling (flagged): 439

Detection summary:
  Suspicious CreationTime changes: 0
  Suspicious ModifiedTime changes: 0
  Anomalous File Creations: 424


Key Findings:
1. LogFile already has MACE timestamps from MFT (no enrichment needed)
2. Timestamp changes are explicitly labeled in Event column
3. Before/after values are in Detail column (need parsing)
4. File system tunneling is partially pre-flagged
5. LogFile complements UsnJrnl (different event coverage)
